In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("Milestone Project").getOrCreate()

In [2]:
from google.colab import files

uploaded=files.upload()

Saving payments.csv to payments.csv
Saving plans.json to plans.json
Saving usage.csv to usage.csv
Saving customers.csv to customers.csv


In [43]:
customers_df= spark.read.csv("customers.csv",header=True,inferSchema=True)

usage_df =spark.read.csv("usage.csv",header=True,inferSchema=True)

payments_df=spark.read.csv("payments.csv",header=True,inferSchema=True)

plans_df=spark.read.option("multiLine","true").json("plans.json")

In [35]:
customers_df.printSchema()

usage_df.printSchema()

payments_df.printSchema()

plans_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [36]:
print("Customer.csv :",customers_df.count())

print("usage.csv :",usage_df.count())

print("Payments.csv :",payments_df.count())

print("Plans.json :",plans_df.count())

Customer.csv : 12
usage.csv : 15
Payments.csv : 15
Plans.json : 4


In [37]:
customers_df.write.mode("overwrite").parquet("bronze/customers")

usage_df.write.mode("overwrite").parquet("bronze/usage")

payments_df.write.mode("overwrite").parquet("bronze/payments")

plans_df.write.mode("overwrite").parquet("bronze/plans")

In [49]:
customers_df.show()

usage_df.show()

payments_df.show()

plans_df.show(truncate=False)

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|           Complete|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|           Complete|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|           Complete|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|           Complete|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|           Complete|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|           Complete|
|        108|   Meer

In [ ]:
from pyspark.sql.functions import *

customers_df.filter(col("plan_id").isNull()).show()

usage_df.filter(col("data_used_gb").isNull()).show()

payments_df.filter(col("amount_paid").isNull()).show()

payments_df.filter(col("payment_mode").isNull()).show()

In [ ]:
customers_df=customers_df.na.fill({"plan_id":"Unknown"})
customers_df.show()

usage_df=usage_df.fillna({"data_used_gb":0})
usage_df.show()

payments_df=payments_df.na.fill({
    "amount_paid":0,
    "payment_mode":"Not Provided"
  })
payments_df.show()

In [ ]:
customers_df = customers_df.withColumn(
    "data_quality_status",when(
        col("plan_id")=="Unknown","Incomplete"
    ).otherwise("Complete")
)
customers_df.show()

usage_df=usage_df.withColumn(
    "data_quality_status",when(
        col("data_used_gb")==0,"Incomplete"
    ).otherwise("Complete")
)
usage_df.show()

payments_df=payments_df.withColumn(
    "data_quality_status",when(
       ( col("amount_paid")==0) | (col("payment_mode")=="Not Provided"),
        "Incomplete"
    ).otherwise("Complete")
)
payments_df.show()

In [ ]:
customers_df.write.mode("overwrite").parquet("silver/customers")

usage_df.write.mode("overwrite").parquet("silver/usage")

payments_df.write.mode("overwrite").parquet("silver/payments")

In [ ]:
flat_df=plans_df.select(
    "plan_id","plan_name","monthly_fee","data_limit_gb",
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming"),
    col("features.unlimited_calls").alias("unlimited_calls"),
    )

flat_df.show()

In [ ]:
flat_df.filter(col("unlimited_calls")=="true").show()

In [ ]:
flat_df.filter(col("ott_included")=="true").show()

In [ ]:
flat_df.select("plan_name","roaming").show()

In [ ]:
flat_df=flat_df.na.fill({
    "roaming":"Not Available"
})

flat_df.show()

In [ ]:
flat_df.write.mode("overwrite").parquet("silver/plans")

In [ ]:
customers_df.join(
    flat_df,"plan_id","left"
).show()

In [ ]:
customers_df.join(
    usage_df,"customer_id","left"
).show()

In [ ]:
customers_df.join(
    payments_df,"customer_id","left"
).show()

In [ ]:
telecom_df = customers_df \
.join(flat_df, "plan_id", "left") \
.join(usage_df, "customer_id", "left") \
.join(payments_df, ["customer_id"], "left")

telecom_df.show()

In [ ]:
customers_df.join(
    flat_df,"plan_id","left_anti"
).show()

In [ ]:
customers_df.join(
    usage_df,"customer_id","left_anti"
).show()

In [ ]:
customers_df.join(
    payments_df,"customer_id","left_anti"
).show()

In [ ]:
usage_df=usage_df.withColumn(
    "usage_category",when(
        col("data_used_gb")>=70,"Heavy user"
    ).when(
        col("data_used_gb")>=30,"Medium user"
    ).otherwise("Low user")
)

usage_df.show()

In [ ]:
payments_df= payments_df.withColumn(
    "payment_category",when(
        col("amount_paid")>=1000,"High payment"
    ).when(
        col("amount_paid")>=500,"Medium payment"
    ).otherwise("Low payment")
)

payments_df.show()

In [ ]:
telecom_df = telecom_df.withColumn(
    "churn_risk",when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    ).when(
        col("data_used_gb") < 15,
        "Medium Risk"
    )
    .otherwise("Low Risk")
)

telecom_df.show()

In [ ]:
telecom_df = telecom_df.withColumn(
    "over_usage_gb",col("data_used_gb") - col("data_limit_gb")
)

telecom_df.show()

In [ ]:
telecom_df = telecom_df.withColumn(
    "over_usage_flag",when(
        col("over_usage_gb") > 0,"Yes"
    ).otherwise("No")
)

telecom_df.show()

In [ ]:
customers_df.groupBy("city").count().show()

In [ ]:
customers_df.groupBy("state").count().show()

In [ ]:
telecom_df.groupBy("plan_name").count().show()

In [ ]:
usage_df.groupBy("usage_category").count().show()

In [ ]:
telecom_df.groupBy("churn_risk").count().show()

In [ ]:
telecom_df.groupBy("plan_name").agg(sum("data_used_gb")).show()

In [ ]:
telecom_df.groupBy("plan_name").agg(avg("data_used_gb")).show()

In [ ]:
telecom_df.groupBy("city").agg(sum("call_minutes")).show()

In [ ]:
telecom_df.groupBy("state").agg(sum("sms_count")).show()

In [ ]:
telecom_df.filter(
    col("payment_status")=="Success"
    ).agg(sum("amount_paid")
).show()

In [ ]:
telecom_df.groupBy("city").agg(sum("amount_paid")).show()

In [ ]:
telecom_df.groupBy("plan_name").agg(sum("amount_paid")).show()

In [ ]:
telecom_df.groupBy("payment_mode").agg(sum("amount_paid")).show()

In [ ]:
telecom_df.groupBy("plan_name").agg(
    sum("amount_paid").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(1)

In [ ]:
telecom_df.groupBy("city").agg(
    sum("amount_paid").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(1)

In [ ]:
from pyspark.sql.window import Window

window_spec= Window.orderBy(
    col("data_used_gb").desc()
)

telecom_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

In [ ]:
window_spec= Window.orderBy(
    col("amount_paid").desc()
)

telecom_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

In [ ]:
window_spec = Window.orderBy(col("data_used_gb").desc())

telecom_df.withColumn(
    "rank",
    rank().over(window_spec)
).show(3)

In [ ]:
customer_revenue = telecom_df.groupBy(
    "customer_id",
    "customer_name").agg(
    sum("amount_paid").alias("total_payment")
)

window_spec = Window.orderBy(col("total_payment").desc())

customer_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).show(3)

In [ ]:
customer_city = telecom_df.groupBy(
    "city",
    "customer_id",
    "customer_name").agg(
    sum("amount_paid").alias("total_payment")
)

window_spec = Window.partitionBy("city").orderBy(
    col("total_payment").desc())

customer_city.withColumn(
    "rank",
    rank().over(window_spec)
).show(1)

In [ ]:
customer_plan = telecom_df.groupBy(
    "plan_name",
    "customer_id",
    "customer_name").agg(
    sum("amount_paid").alias("total_payment")
)

window_spec = Window.partitionBy("plan_name").orderBy(
    col("total_payment").desc())

customer_plan.withColumn(
    "rank",
    rank().over(window_spec)
).show(1)

In [ ]:
monthly = telecom_df.groupBy(
    "bill_month").agg(
    sum("amount_paid").alias("monthly_revenue")
)

window_spec = Window.orderBy("bill_month")

monthly.withColumn(
    "running_total",
    sum("monthly_revenue").over(window_spec)
).show()

In [ ]:
window_spec = Window.partitionBy(
    "customer_id"
).orderBy("usage_month")

telecom_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb").over(window_spec)
).show()

In [ ]:
window_spec = Window.partitionBy(
    "customer_id").orderBy("usage_month")

telecom_df.withColumn(
    "next_month_usage",
    lead("data_used_gb").over(window_spec)
).show()

In [ ]:
window_spec = Window.partitionBy(
    "customer_id").orderBy("usage_month")

usage_growth = telecom_df.withColumn(
    "previous_usage",
    lag("data_used_gb").over(window_spec))

usage_growth.filter(
    col("data_used_gb") > col("previous_usage")).select(
    "customer_id",
    "customer_name",
    "usage_month",
    "previous_usage",
    "data_used_gb"
).show()

In [ ]:
customers_df.createOrReplaceTempView("customers")

usage_df.createOrReplaceTempView("usage")

payments_df.createOrReplaceTempView("payments")

flat_df.createOrReplaceTempView("plans")

In [ ]:
spark.sql("""
select * from customers
where status="Active"
""").show()

In [ ]:
spark.sql("""
select city,count(*) from customers
group by city
""").show()

In [ ]:
spark.sql("""
select
  p.plan_name,
  SUM(pay.amount_paid) AS total_revenue
  FROM customers c
LEFT JOIN plans p
ON c.plan_id=p.plan_id
LEFT JOIN payments pay
ON c.customer_id=pay.customer_id
GROUP BY p.plan_name
""").show()

In [ ]:
spark.sql("""
SELECT
c.customer_id,
c.customer_name,
u.data_used_gb
FROM customers c
JOIN usage u
ON c.customer_id=u.customer_id
WHERE usage_category="Heavy user"
""").show()

In [ ]:
telecom_df.createOrReplaceTempView("telecom")

In [ ]:
spark.sql("""
select customer_id,customer_name from telecom
where churn_risk="High Risk"
""").show()

In [ ]:
spark.sql("""
SELECT *
FROM customers
WHERE plan_id='Unknown'
""").show()

In [ ]:
spark.sql("""
SELECT *
FROM payments
WHERE payment_status IN ('Failed','Pending')
""").show()

In [ ]:
spark.sql("""
SELECT
customer_id,
customer_name,
data_used_gb
FROM telecom
ORDER BY data_used_gb DESC
LIMIT 5
""").show()

In [ ]:
spark.sql("""
SELECT
payment_mode,
SUM(amount_paid) AS total_revenue
FROM payments
GROUP BY payment_mode
""").show()

In [ ]:
telecom_df.write.mode("overwrite").parquet("gold/telecom")

In [ ]:
telecom_df.write.mode("overwrite").partitionBy("usage_month") \
.parquet("gold_partitioned")

In [ ]:
%%writefile usage_incremental.csv

usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,55,1050,135
1017,102,2026-03,40,720,95
1018,104,2026-03,62,1250,170
1019,108,2026-03,90,1800,300
1020,109,2026-03,50,980,110

In [ ]:
increment_df = spark.read.csv("usage_incremental.csv",header=True,
    inferSchema=True)

increment_df.show()

In [ ]:
increment_df.write.mode("append").parquet("silver/usage")

In [ ]:
silver_usage = spark.read.parquet("silver/usage")

silver_usage.groupBy("customer_id").agg(
    sum("data_used_gb").alias("total_usage")
).show()

In [ ]:
updated_usage = spark.read.parquet(
    "silver/usage"
)

gold_df = customers_df \
.join(flat_df,"plan_id","left") \
.join(updated_usage,"customer_id","left") \
.join(payments_df,"customer_id","left")

gold_df.write \
.mode("overwrite") \
.partitionBy("usage_month") \
.parquet("gold_partitioned")

In [ ]:
print("Original Usage Count :",usage_df.count())

print("Updated Usage Count :",updated_usage.count())

In [ ]:
customer_usage_summary = telecom_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
)

customer_usage_summary.show()

In [ ]:

from pyspark.sql.functions import countDistinct
plan_report = telecom_df.groupBy(
    "plan_name"
).agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage"),
    sum("amount_paid").alias("total_revenue")
)

plan_report.show()

In [ ]:
city_report = telecom_df.groupBy(
    "city"
).agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("amount_paid").alias("total_revenue"),
    avg("amount_paid").alias("average_payment")
)

city_report.show()

In [ ]:
churn_report = telecom_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
)

churn_report.show()

In [161]:
over_usage_report = telecom_df.select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
).filter(
    col("over_usage_flag")=="Yes"
)

over_usage_report.show()

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|  plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
+-----------+-------------+-----------+------------+-------------+-------------+

